# 02 — CrafText adapter boundary
Normalize a CrafText-shaped `reset/step` API into terminated, truncated and a static action mask.

In [ ]:
from dataclasses import dataclass
import numpy as np
from tunix_craftext.adapters import CrafTextAdapter


In [ ]:
@dataclass(frozen=True)
class State:
    timestep: int

class TinyCrafText:
    def reset(self, key, params):
        return np.array([key, 0]), State(0)
    def step(self, key, state, action, params):
        next_state = State(state.timestep + 1)
        return np.array([next_state.timestep, action]), next_state, np.array(float(action)), np.array(next_state.timestep >= 2), {'action_mask': np.array([True, action == 0, True])}


In [ ]:
adapter = CrafTextAdapter(TinyCrafText(), params=object(), action_count=3)
reset = adapter.reset(key=42)
transition = adapter.step(key=43, state=reset.state, action=0)
print('initial mask:', reset.action_mask)
print('terminated / truncated:', transition.terminated, transition.truncated)
print('next mask:', transition.action_mask)


Replace `TinyCrafText` with a vendor world-preset environment after installing the `envs` extra. Keep setup outside JIT/scan; the adapter result is the training boundary.